In [4]:
import requests
import pandas as pd

COUNTIES = [
    {"county": "San Francisco", "state": "CA", "fips": "06075", "lat": 37.7749, "lon": -122.4194,},
    {"county": "San Mateo", "state": "CA", "fips": "06081", "lat": 37.5630, "lon": -122.3255,},
    {"county": "Santa Cruz", "state": "CA", "fips": "06087", "lat": 36.9741, "lon": -122.0308,},
    {"county": "San Luis Obispo", "state": "CA", "fips": "06079", "lat": 35.2828, "lon": -120.6596,},
    {"county": "Monterey", "state": "CA", "fips": "06053", "lat": 36.6002, "lon": -121.8947,},
    {"county": "Santa Barbara", "state": "CA", "fips": "06083", "lat": 34.4208, "lon": -119.6982,},
    {"county": "Ventura","state": "CA", "fips": "06111", "lat": 34.2805, "lon": -119.2945,},
    {"county": "Los Angeles", "state": "CA", "fips": "06037", "lat": 34.0522, "lon": -118.2437,},
    {"county": "Orange", "state": "CA", "fips": "06059", "lat": 33.7175, "lon": -117.8311,},
    {"county": "San Diego", "state": "CA", "fips": "06073", "lat": 32.8668, "lon": -117.1410,},
]

HEADERS = {"User-Agent": "crisis-responder-demo/0.1 (educational portfolio project)"} 

def get_forecastURL(lat, lon): 
    '''
    convert latitude and longitude into a NWS forecast endpoint
    gets hourly weather 
    ''' 

    url = f"https://api.weather.gov/points/{lat},{lon}" # build url
    response = requests.get(url, headers = HEADERS, timeout = 30) # location metadata 
    response.raise_for_status() 
    data = response.json() 

    if "properties" not in data: 
        print("Unexpected Response:")
        print(data)
        raise ValueError("No properties field found in NWS response") 

    return data["properties"]["forecastHourly"] # extract hourly forecast 


def get_hourly_weather(county): 
    '''
    get hourly forecast data per county
    ''' 

    forecast_url = get_forecastURL(county["lat"], county["lon"]) # get hourly forecast url 
    response = requests.get(forecast_url, headers = HEADERS, timeout = 30) # actual forecast 
    response.raise_for_status() 

    periods = response.json()["properties"]["periods"] # hourly forecast periods 

    # iterate through each period 
    rows = [] 
    for p in periods: 
        rows.append({ 
            "county": county["county"], 
            "state": county["state"], 
            "county_fips": county["fips"], 
            "timestamp": p.get("startTime"), 
            "temperature_f": p.get("temperature"), 
            "wind_speed_raw": p.get("windSpeed"), 
            "short_forecast": p.get("shortForecast")
        }) 

    return rows 

def main(): 
    all_rows = [] 

    for county in COUNTIES: 
        try: 
            rows = get_hourly_weather(county)
            all_rows.extend(rows) 
        except Exception as e: 
            print(f"Failed for {county['county']}: {e}") 

    df = pd.DataFrame(all_rows) 
    df.to_csv("weather_hourly.csv", index = False) 
    print(df['county'].value_counts()) 

if __name__ == "__main__":
    main()
    

county
San Francisco      156
San Mateo          156
Santa Cruz         156
San Luis Obispo    156
Monterey           156
Santa Barbara      156
Ventura            156
Los Angeles        156
Orange             156
San Diego          156
Name: count, dtype: int64


In [19]:
import pandas as pd 
import re 

# convert text fields into numeric fields 

df = pd.read_csv("weather_hourly.csv") 

# wind speed 
def parse_wind_speed(value):
    if pd.isna(value):
        return None

    nums = re.findall(r"\d+", str(value))

    if len(nums) == 0:
        return None

    nums = [int(n) for n in nums]

    if len(nums) == 1:
        return nums[0]

    return sum(nums) / len(nums)

df["wind_mph"] = df["wind_speed_raw"].apply(parse_wind_speed)

# Make numeric columns safer
df["temperature_f"] = pd.to_numeric(df["temperature_f"], errors="coerce")
df["wind_mph"] = pd.to_numeric(df["wind_mph"], errors="coerce")

# date time stuff 
df['timestamp'] = pd.to_datetime(df["timestamp"], errors = 'coerce') 
df['date'] = df['timestamp'].dt.date 

# groupby to go from hourly weather to daily 
weather_daily = (
    df.groupby(["county_fips", "county", "state", "date"], as_index=False)
      .agg(
          avg_temp_f=("temperature_f", "mean"),
          max_temp_f=("temperature_f", "max"),
          avg_wind_mph=("wind_mph", "mean"),
          max_wind_mph=("wind_mph", "max"),
      )
)

weather_daily.to_csv("weather_clean.csv", index=False)

   county_fips       county state        date  avg_temp_f  max_temp_f  \
0         6037  Los Angeles    CA  2026-05-07   68.333333          77   
1         6037  Los Angeles    CA  2026-05-08   65.500000          77   
2         6037  Los Angeles    CA  2026-05-09   68.541667          82   
3         6037  Los Angeles    CA  2026-05-10   69.291667          83   
4         6037  Los Angeles    CA  2026-05-11   71.583333          86   

   avg_wind_mph  max_wind_mph  
0      7.777778            10  
1      3.541667            10  
2      3.750000            10  
3      3.750000            10  
4      3.125000            10  
county_fips     0
county          0
state           0
date            0
avg_temp_f      0
max_temp_f      0
avg_wind_mph    0
max_wind_mph    0
dtype: int64
(80, 8)


In [20]:
# Simulate humidity based on monthly averages 
import pandas as pd
import numpy as np 

df = pd.read_csv("weather_clean.csv") 

np.random.seed(42)

avg_humidity = { 
    "San Francisco": 75, 
    "San Mateo": 75, 
    "Santa Cruz": 77, 
    "Monterey": 75, 
    "Santa Barbara": 67, 
    "Ventura": 66, 
    "Los Angeles": 65, 
    "Orange": 67, 
    "San Diego": 69
} 


def simulate_humidity(row): 
    # get baseline values 
    baseline = avg_humidity.get(row["county"], 65)

    # add daily variation around county average 
    simulate_value = np.random.normal(loc = baseline, scale = 10) 

    # keep humidity values realisitc 
    return round(np.clip(simulate_value, 5, 95), 1)

# add humidity column to data frame 
df["humidity"] = df.apply(simulate_humidity, axis = 1) 

# add missing values to mimic real missing values on purpose 
missing_idx = df.sample(frac = 0.10, random_state = 42).index 
df.loc[missing_idx, "humidity"] = None 

df.to_csv("weather_clean.csv", index = False) 
